# Data Splitting

Data splitting is one of the most fundamental and critical steps in the supervised machine learning process. Before training a model, it is imperative to divide the available dataset into several distinct subsets, each playing a specific role in the model development cycle. Without this rigorous partitioning, it is impossible to honestly assess the generalization capabilities of a model — that is, its ability to perform correctly on data it has never seen.

The `DataSplitter` class implements **4 methods** covering the most common situations encountered in practice:

| Method | Target situation |
|---|---|
| `train_test_split` | General case, data without any particular order |
| `stratified_train_test_split` | Imbalanced classes (e.g. 90% vs 10%) |
| `temporal_train_test_split` | Time series (data ordered over time) |
| `k_fold_split` | Robust evaluation via cross-validation |

## 1. Key Concepts

### Why split data?

A machine learning model learns by adjusting its internal parameters to minimize an error on the training data. If we evaluate that same model on the data used for training, we obtain a biased measure: the model has already "seen" these examples, and its results on them do not reflect its real ability to handle new situations.

Splitting simulates deployment: we reserve a portion of the data, which the model will never see during training, to measure its performance under conditions close to reality.

### The Train / Validation / Test triplet

When dividing a dataset, we generally create two or three subsets. Each serves a distinct and non-interchangeable role.

**Training set (Train)**

This is the portion on which the model learns, discovering the relationships between variables. The larger and more representative it is, the better the quality of learning.
**Analogy**: the coursework and exercises done by a student.

**Validation set (Validation)**

It is used to tune hyperparameters and compare several configurations, without touching the test set. It allows the model to be optimized before the final evaluation.
**Analogy**: mock exams to practice before the official test.

#####**Test set (Test)**

This is the final, objective measure, used only once to estimate the model's real performance under new conditions.
**Analogy**: the final exam, which reflects the student's true level.

### Data Leakage

Data leakage is one of the most dangerous errors in machine learning, precisely because it produces **falsely good results**: the model appears to perform well during evaluation, but fails in production on real data.

It occurs when information from the test set "contaminates" the training of the model. The model then indirectly learns elements it should not know in advance.

**Concrete example:** Suppose we want to normalize the data before splitting. To compute this normalization, we use the mean and standard deviation of the entire dataset, including the test set. As a result, the model has already absorbed residual information about the test set before being evaluated on it.

- Bad practice: normalize BEFORE splitting
- Good practice: split FIRST, then compute normalization only on the training set

Rigorous splitting is therefore the first line of defense against data leakage.

### The Random Seed

Splitting randomly shuffles the data before cutting it. This raises a practical problem: if we re-run the code, the shuffle will be different, and the splits will no longer be the same. Results will therefore not be reproducible.

The **seed** is an integer we fix to "freeze" the randomness. By providing the same seed, NumPy will always generate the same pseudo-random sequence, and therefore the same split.

It is essential for:
- Reproducing an experiment and obtaining exactly the same results
- Fairly comparing two methods (they see exactly the same data)
- Sharing work with other researchers or developers

## 2. The 4 Splitting Methods

Before addressing the implementation, it is essential to intuitively understand each splitting method: its principles, use cases, advantages, and limitations. This section is entirely theoretical — no code appears here. The reader should be able to choose the appropriate method for their problem before even looking at the code.


### 2.1 Simple random splitting — `train_test_split`

This method shuffles all the data and then cuts it according to a fixed ratio (for example 80% for training and 20% for testing). It is widely used because it is simple and fast, suitable for general cases such as image classification or numerical value prediction. Its main drawback is that it does not guarantee proportional representation of classes and is not suitable for time series.

In practice, it is often the first choice when you want to quickly test a model. The reproducibility of the shuffle can be controlled through a random seed, which allows several trials to be compared under identical conditions.

### 2.2 Stratified splitting — `stratified_train_test_split`

**Stratified splitting** preserves the proportions of each class by first separating the data by category before applying the ratio. This allows minority classes to be better represented and a more reliable evaluation to be obtained, particularly in imbalanced datasets such as fraud detection or medical diagnostics. However, this method only applies to classification problems, as it requires a categorical target variable.

This approach is particularly useful for preventing a rare class from disappearing entirely from the test set. It ensures that even minority classes are present in both sets, making the evaluation more realistic.

### 2.3 Temporal splitting — `temporal_train_test_split`

When working with chronological data, it is essential to respect the order of time. Temporal splitting consists of keeping the oldest observations for training and reserving the most recent for testing. Unlike random methods, there is no shuffling: the boundary is set by the chronology. This approach is indispensable for time series such as stock prices, weather forecasts, or daily sales.

This method avoids any information leakage, as the model never sees the future during training. It is simple to apply but can give a limited evaluation if the chosen boundary does not well reflect the variability of the data.

### 2.4 K-Fold cross-validation — `k_fold_split`

**K-Fold cross-validation** divides the data into **K** equal-sized subsets, then repeats **K** times the training and testing: at each iteration, a different fold serves as the test set and the others as the training set. This yields an average of performances, making the evaluation more robust. This technique is valuable for small datasets, as it exploits all observations.

Its **main advantage** is to reduce the risk that a particular split unduly influences the result. However, it requires more computation time and remains unsuitable for temporal data, where order must be preserved.

### Comparative summary of the 4 methods

| Criterion | Train-Test | Stratified | Temporal | K-Fold |
|---|---|---|---|---|
| Random shuffling | Yes | Yes (per class) | No | Yes |
| Preserves proportions | Not guaranteed | Yes | — | Not guaranteed |
| Respects temporal order | No | No | Yes | No |
| Number of splits | 1 | 1 | 1 | k |
| Uses all data | No | No | No | Yes |
| Main use case | General | Imbalanced classes | Time series | Robust evaluation |

## 3. Pseudo-algorithm

The four methods of `DataSplitter` follow a common logic, although each introduces important variations. Here is their unified algorithmic formulation.

**Simple random splitting (`train_test_split`):**

1. $\mathcal{D} \leftarrow$ dataset of $n$ observations $(\mathbf{x}_i, y_i)$
2. $\mathcal{I} \leftarrow [0, 1, \ldots, n-1]$ — index array
3. Shuffle $\mathcal{I}$ randomly (according to the fixed seed)
4. $n_{\text{test}} \leftarrow \lfloor \text{test\_size} \times n \rfloor$
5. $\mathcal{I}_{\text{test}} \leftarrow \mathcal{I}[0 : n_{\text{test}}]$, $\quad \mathcal{I}_{\text{train}} \leftarrow \mathcal{I}[n_{\text{test}} : n]$
6. **return** $\mathbf{X}[\mathcal{I}_{\text{train}}],\ \mathbf{X}[\mathcal{I}_{\text{test}}],\ \mathbf{y}[\mathcal{I}_{\text{train}}],\ \mathbf{y}[\mathcal{I}_{\text{test}}]$

**Stratified splitting (`stratified_train_test_split`):**

1. $\mathcal{C} \leftarrow$ list of unique classes in $\mathbf{y}$
2. **for each** class $c \in \mathcal{C}$ **do**
3. $\quad \mathcal{I}_c \leftarrow \{i : y_i = c\}$ — indices of class $c$
4. $\quad$ Shuffle $\mathcal{I}_c$
5. $\quad$ Split $\mathcal{I}_c$ according to $\text{test\_size}$ $\rightarrow \mathcal{I}_{c,\text{test}}$ and $\mathcal{I}_{c,\text{train}}$
6. $\quad$ Accumulate into $\mathcal{I}_{\text{test}}$ and $\mathcal{I}_{\text{train}}$
7. **return** the splits built from the accumulated indices

**Temporal splitting (`temporal_train_test_split`):**

1. No shuffling — chronological order must be preserved
2. $n_{\text{test}} \leftarrow \lfloor \text{test\_size} \times n \rfloor$
3. $\mathcal{D}_{\text{test}} \leftarrow \mathcal{D}[-n_{\text{test}} :]$ — most recent data
4. $\mathcal{D}_{\text{train}} \leftarrow \mathcal{D}[: -n_{\text{test}}]$ — oldest data
5. **return** $\mathcal{D}_{\text{train}},\ \mathcal{D}_{\text{test}}$

**K-Fold cross-validation (`k_fold_split`):**

1. Shuffle $\mathcal{I} = [0, 1, \ldots, n-1]$
2. $s \leftarrow \lfloor n / k \rfloor$ — size of each fold
3. **for** $i \in [0, k-1]$ **do**
4. $\quad \mathcal{I}_{\text{test}}^{(i)} \leftarrow \mathcal{I}[i \cdot s : (i+1) \cdot s]$
5. $\quad \mathcal{I}_{\text{train}}^{(i)} \leftarrow \mathcal{I} \setminus \mathcal{I}_{\text{test}}^{(i)}$
6. $\quad$ Append $(\mathcal{I}_{\text{train}}^{(i)},\ \mathcal{I}_{\text{test}}^{(i)})$ to the list of folds
7. **return** the list of $k$ pairs $(\text{train}, \text{test})$

## 4. Implementation

For this implementation, we will use a synthetic dataset built from scratch, allowing us to precisely control the properties of the data: size, number of classes, imbalance, and temporal dimension. Each method is tested and compared along two axes — **scikit-learn** and **IFRI MiniLib** — with execution time measurement and a unified results table.


In [1]:
import pandas as pd
import numpy as np
import time
from ifri_mini_ml_lib.preprocessing.preparation.splitting import DataSplitter

# ── Building the demo datasets ──────────────────────────────────────────────
np.random.seed(42)
n = 200

X = pd.DataFrame({
    'feature_1': np.random.randn(n),
    'feature_2': np.random.randn(n),
    'feature_3': np.random.rand(n) * 100
})

# Balanced labels (3 classes)
y_balanced = pd.Series(np.random.randint(0, 3, n))

# Imbalanced labels (10% class 1 — rare case, e.g. fraud)
y_imbalanced = pd.Series(np.where(np.random.rand(n) < 0.1, 1, 0))

# Temporal dataset: index = dates
dates = pd.date_range('2022-01-01', periods=n, freq='D')
X_temporal = X.copy()
X_temporal.index = dates
y_temporal = pd.Series(np.cumsum(np.random.randn(n)), index=dates)

print(f"Main dataset        : {n} observations, {X.shape[1]} features")
print(f"Balanced labels     : {dict(y_balanced.value_counts().sort_index())}")
print(f"Imbalanced labels   : {dict(y_imbalanced.value_counts().sort_index())}")
print(f"Temporal period     : {dates[0].date()} → {dates[-1].date()}")

Main dataset        : 200 observations, 3 features
Balanced labels     : {0: np.int64(69), 1: np.int64(60), 2: np.int64(71)}
Imbalanced labels   : {0: np.int64(175), 1: np.int64(25)}
Temporal period     : 2022-01-01 → 2022-07-19


In [2]:
X.head()

,feature_1,feature_2,feature_3
0,0.496714,0.357787,41.481950
1,-0.138264,0.560785,27.340707
2,0.647689,1.083051,5.637550
3,1.523030,1.053802,86.472238
4,-0.234153,-1.377669,81.290101


---

### 4.1 `train_test_split` — Simple random splitting

#### With scikit-learn


In [3]:
from sklearn.model_selection import train_test_split as sk_tts

start_1 = time.perf_counter()
X_train_sk, X_test_sk, y_train_sk, y_test_sk = sk_tts(
    X, y_balanced, test_size=0.2, random_state=42
)
end_1 = time.perf_counter()

#### With ifri-mini-ml-lib


In [4]:
splitter_tts = DataSplitter(seed=42)

start_2 = time.perf_counter()
X_train_ifri, X_test_ifri, y_train_ifri, y_test_ifri = splitter_tts.train_test_split(
    X, y_balanced, test_size=0.2
)
end_2 = time.perf_counter()

In [5]:
# ── Comparison metrics ───────────────────────────────────────────────────────
def class_ratio(y, cls=0):
    return (y == cls).sum() / len(y)

tts_results = pd.DataFrame({
    'Metric': [
        'Train size',
        'Test size',
        'Test ratio (%)',
        'Class 0 ratio — train (%)',
        'Class 0 ratio — test (%)',
        'Execution time (ms)',
    ],
    'scikit-learn': [
        len(X_train_sk),
        len(X_test_sk),
        round(len(X_test_sk) / n * 100, 1),
        round(class_ratio(y_train_sk, 0) * 100, 1),
        round(class_ratio(y_test_sk,  0) * 100, 1),
        round((end_1 - start_1) * 1000, 4),
    ],
    'ifri-mini-ml-lib': [
        len(X_train_ifri),
        len(X_test_ifri),
        round(len(X_test_ifri) / n * 100, 1),
        round(class_ratio(y_train_ifri, 0) * 100, 1),
        round(class_ratio(y_test_ifri,  0) * 100, 1),
        round((end_2 - start_2) * 1000, 4),
    ],
})

In [6]:
tts_results.T

,0,1,2,3,4,5
Metric,Train size,Test size,Test ratio (%),Class 0 ratio — train (%),Class 0 ratio — test (%),Execution time (ms)
scikit-learn,160.0,40.0,20.0,35.6,30.0,3.2775
ifri-mini-ml-lib,160.0,40.0,20.0,35.6,30.0,2.0262


---

### 4.2 `stratified_train_test_split` — Stratified splitting

#### With scikit-learn


In [7]:
start_3 = time.perf_counter()
X_train_sk_s, X_test_sk_s, y_train_sk_s, y_test_sk_s = sk_tts(
    X, y_imbalanced, test_size=0.2, random_state=42, stratify=y_imbalanced
)
end_3 = time.perf_counter()

#### With ifri-mini-ml-lib


In [8]:
splitter_strat = DataSplitter(seed=42)

start_4 = time.perf_counter()
X_train_ifri_s, X_test_ifri_s, y_train_ifri_s, y_test_ifri_s = splitter_strat.stratified_train_test_split(
    X, y_imbalanced, test_size=0.2
)
end_4 = time.perf_counter()

In [9]:
# ── Comparison metrics ───────────────────────────────────────────────────────
pct_ref = (y_imbalanced == 1).sum() / len(y_imbalanced) * 100

strat_results = pd.DataFrame({
    'Metric': [
        'Train size',
        'Test size',
        'Test ratio (%)',
        '% class 1 original',
        '% class 1 — train',
        '% class 1 — test',
        'Execution time (ms)',
    ],
    'scikit-learn': [
        len(X_train_sk_s),
        len(X_test_sk_s),
        round(len(X_test_sk_s) / n * 100, 1),
        round(pct_ref, 1),
        round((y_train_sk_s == 1).sum() / len(y_train_sk_s) * 100, 1),
        round((y_test_sk_s  == 1).sum() / len(y_test_sk_s)  * 100, 1),
        round((end_3 - start_3) * 1000, 4),
    ],
    'ifri-mini-ml-lib': [
        len(X_train_ifri_s),
        len(X_test_ifri_s),
        round(len(X_test_ifri_s) / n * 100, 1),
        round(pct_ref, 1),
        round((y_train_ifri_s == 1).sum() / len(y_train_ifri_s) * 100, 1),
        round((y_test_ifri_s  == 1).sum() / len(y_test_ifri_s)  * 100, 1),
        round((end_4 - start_4) * 1000, 4),
    ],
})

In [10]:
strat_results.T

,0,1,2,3,4,5,6
Metric,Train size,Test size,Test ratio (%),% class 1 original,% class 1 — train,% class 1 — test,Execution time (ms)
scikit-learn,160.0,40.0,20.0,12.5,12.5,12.5,6.0907
ifri-mini-ml-lib,160.0,40.0,20.0,12.5,12.5,12.5,3.8073


---

### 4.3 `k_fold_split` — K-Fold cross-validation

#### With scikit-learn


In [11]:
from sklearn.model_selection import KFold

kf_sk = KFold(n_splits=5, shuffle=True, random_state=42)

start_5 = time.perf_counter()
folds_sk = [(X.iloc[tr], X.iloc[te], y_balanced.iloc[tr], y_balanced.iloc[te])
            for tr, te in kf_sk.split(X)]
end_5 = time.perf_counter()

#### With ifri-mini-ml-lib


In [12]:
splitter_kf = DataSplitter(seed=42)

start_6 = time.perf_counter()
folds_ifri = splitter_kf.k_fold_split(X, y_balanced, k=5)
end_6 = time.perf_counter()

In [13]:
# ── Comparison metrics (averaged over 5 folds) ──────────────────────────────
train_sizes_sk   = [len(f[0]) for f in folds_sk]
test_sizes_sk    = [len(f[1]) for f in folds_sk]
train_sizes_ifri = [len(f[0]) for f in folds_ifri]
test_sizes_ifri  = [len(f[1]) for f in folds_ifri]

kf_results = pd.DataFrame({
    'Metric': [
        'Number of folds',
        'Train size (avg.)',
        'Test size (avg.)',
        'Test ratio — fold 1 (%)',
        'Test ratio — fold 2 (%)',
        'Test ratio — fold 3 (%)',
        'Test ratio — fold 4 (%)',
        'Test ratio — fold 5 (%)',
        'Execution time (ms)',
    ],
    'scikit-learn': [
        len(folds_sk),
        round(np.mean(train_sizes_sk), 1),
        round(np.mean(test_sizes_sk),  1),
        round(test_sizes_sk[0] / n * 100, 1),
        round(test_sizes_sk[1] / n * 100, 1),
        round(test_sizes_sk[2] / n * 100, 1),
        round(test_sizes_sk[3] / n * 100, 1),
        round(test_sizes_sk[4] / n * 100, 1),
        round((end_5 - start_5) * 1000, 4),
    ],
    'ifri-mini-ml-lib': [
        len(folds_ifri),
        round(np.mean(train_sizes_ifri), 1),
        round(np.mean(test_sizes_ifri),  1),
        round(test_sizes_ifri[0] / n * 100, 1),
        round(test_sizes_ifri[1] / n * 100, 1),
        round(test_sizes_ifri[2] / n * 100, 1),
        round(test_sizes_ifri[3] / n * 100, 1),
        round(test_sizes_ifri[4] / n * 100, 1),
        round((end_6 - start_6) * 1000, 4),
    ],
})

In [14]:
kf_results.T

,0,1,2,3,4,5,6,7,8
Metric,Number of folds,Train size (avg.),Test size (avg.),Test ratio — fold 1 (%),Test ratio — fold 2 (%),Test ratio — fold 3 (%),Test ratio — fold 4 (%),Test ratio — fold 5 (%),Execution time (ms)
scikit-learn,5.0,160.0,40.0,20.0,20.0,20.0,20.0,20.0,9.6428
ifri-mini-ml-lib,5.0,160.0,40.0,20.0,20.0,20.0,20.0,20.0,7.9671


###**Why no comparison for temporal_train_test_split**

              **scikit-learn has no direct equivalent  **

It does not provide a function for a simple temporal split into two parts. Instead, it offers TimeSeriesSplit, which is a cross-validation method and not a bipartition.

**Fundamental difference**

-**temporal_train_test_split** → a single fixed boundary, yielding a train and a test set.

-**TimeSeriesSplit** → several successive folds, used for model selection and hyperparameter tuning.

**Why avoid random shuffling  **

In a time series, shuffling the data breaks causality: the model would see future data during training. This creates a temporal data leakage, producing artificially good but unusable results in production.

**Why no comparison  **

Comparing the two would be misleading: one is a simple split, the other is cross-validation. They are not at the same level of abstraction, hence the deliberate absence of a comparison.

## 5. Interactive Demo


In [15]:
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

def demo_splitting(method, test_size, k_folds, seed):
    # Interactively visualizes the result of each splitting method.
    np.random.seed(0)
    n_demo = 100
    X_d = pd.DataFrame({'x': np.random.randn(n_demo),
                         'y': np.random.randn(n_demo)})
    y_d = pd.Series(np.where(np.random.rand(n_demo) < 0.25, 1, 0))
    dates_d = pd.date_range('2023-01-01', periods=n_demo, freq='D')
    X_d_t = X_d.copy(); X_d_t.index = dates_d

    sp = DataSplitter(seed=seed)
    fig, ax = plt.subplots(figsize=(11, 3))

    if method == 'train_test_split':
        Xtr, Xte, ytr, yte = sp.train_test_split(X_d, y_d, test_size=test_size)
        sizes  = [len(Xtr), len(Xte)]
        labels = [f'Train\n{len(Xtr)} ({len(Xtr)/n_demo:.0%})',
                  f'Test\n{len(Xte)} ({len(Xte)/n_demo:.0%})']
        colors = ['#4CAF50', '#F44336']
        left = 0
        for sz, lbl, col in zip(sizes, labels, colors):
            ax.barh(0, sz, left=left, height=0.5, color=col)
            ax.text(left + sz/2, 0, lbl, ha='center', va='center',
                    fontweight='bold', color='white', fontsize=11)
            left += sz
        ax.set_title('train_test_split — Data distribution', fontweight='bold', fontsize=13)

    elif method == 'stratified_train_test_split':
        Xtr, Xte, ytr, yte = sp.stratified_train_test_split(X_d, y_d, test_size=test_size)
        pct_orig = (y_d == 1).mean() * 100
        pct_tr   = (ytr  == 1).mean() * 100
        pct_te   = (yte  == 1).mean() * 100
        cats = ['Original dataset', 'Train', 'Test']
        p0   = [100 - pct_orig, 100 - pct_tr, 100 - pct_te]
        p1   = [pct_orig, pct_tr, pct_te]
        x_pos = np.arange(3)
        ax.bar(x_pos, p0, color='#4CAF50', label='Class 0')
        ax.bar(x_pos, p1, bottom=p0, color='#F44336', label='Class 1')
        for xi, v0, v1 in zip(x_pos, p0, p1):
            ax.text(xi, v0/2,       f'{v0:.1f}%', ha='center', va='center',
                    color='white', fontweight='bold', fontsize=10)
            ax.text(xi, v0 + v1/2,  f'{v1:.1f}%', ha='center', va='center',
                    color='white', fontweight='bold', fontsize=10)
        ax.set_xticks(x_pos); ax.set_xticklabels(cats)
        ax.set_ylabel('Proportion (%)'); ax.legend()
        ax.set_title('stratified_train_test_split — Preserved proportions', fontweight='bold', fontsize=13)

    elif method == 'temporal_train_test_split':
        Xtr, Xte, ytr, yte = sp.temporal_train_test_split(X_d_t, y_d, test_size=test_size)
        y_plot = pd.Series(np.cumsum(np.random.randn(n_demo)), index=dates_d)
        ax.plot(y_plot[Xtr.index], color='#4CAF50', linewidth=2,
                label=f'Train ({len(Xtr)} pts)')
        ax.plot(y_plot[Xte.index], color='#F44336', linewidth=2,
                label=f'Test ({len(Xte)} pts)')
        ax.axvline(x=Xte.index[0], color='black', linestyle='--', linewidth=2)
        ax.set_xlabel('Date')
        ax.legend()
        ax.set_title('temporal_train_test_split — Chronological split', fontweight='bold', fontsize=13)

    elif method == 'k_fold_split':
        folds = sp.k_fold_split(X_d, y_d, k=k_folds)
        fold_size = n_demo // k_folds
        for i in range(k_folds):
            for j in range(k_folds):
                color = '#F44336' if j == i else '#4CAF50'
                ax.barh(i, fold_size, left=j*fold_size, height=0.6, color=color, edgecolor='white')
                ax.text(j*fold_size + fold_size/2, i,
                        'TEST' if j == i else 'train',
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color='white')
        ax.set_yticks(range(k_folds))
        ax.set_yticklabels([f'Cycle {i+1}' for i in range(k_folds)])
        ax.set_title(f'k_fold_split — Test fold rotation (k={k_folds})', fontweight='bold', fontsize=13)

    ax.set_xlim(0, n_demo if method != 'stratified_train_test_split' else None)
    if method not in ('stratified_train_test_split', 'k_fold_split', 'temporal_train_test_split'):
        ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    demo_splitting,
    method=Dropdown(
        options=['train_test_split', 'stratified_train_test_split',
                 'temporal_train_test_split', 'k_fold_split'],
        value='train_test_split',
        description='Method:'
    ),
    test_size=FloatSlider(min=0.1, max=0.4, step=0.05, value=0.2, description='test_size:'),
    k_folds=IntSlider(min=2, max=10, step=1, value=5, description='k (K-Fold):'),
    seed=IntSlider(min=0, max=100, step=1, value=42, description='Seed:'),
);

interactive(children=(Dropdown(description='Method:', options=('train_test_split', 'stratified_train_test_spli…

The interactive demo above allows you to visualize the result of each splitting method for different parameters. You can adjust the test proportion (`test_size`), the number of folds (`k`), and the random seed (`seed`) to observe their effect on the data distribution. For the stratified method, the chart shows the preservation of class proportions; for K-Fold, the rotation of the test fold at each cycle; for the temporal split, the chronological boundary.

## 6. Real-world Applications

Data splitting is a universal step, present in every machine learning project, regardless of the domain and the nature of the data. The methods implemented in `DataSplitter` cover the most common cases encountered in practice.

1. **Finance and fraud detection**: In banking fraud detection systems, fraudulent transactions represent less than 1% of the data. Stratified splitting is essential to ensure that the test set contains enough fraud examples for a reliable evaluation. Without stratification, a model could appear excellent by achieving 99% accuracy simply by predicting "no fraud" — without ever detecting a single fraud. (e.g.: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

2. **Economic forecasts and financial markets**: Predicting stock prices, exchange rates, or energy demand relies on chronological data. Temporal splitting is mandatory here: training a model on 2020-2024 data to predict 2019 prices would constitute obvious data leakage. The chronological boundary is the only guarantee of evaluation integrity.

3. **Healthcare and medical diagnosis**: Evaluating a diagnostic model for rare diseases (e.g. a disease affecting 2% of the population) requires stratification so that each partition contains representative examples. K-Fold cross-validation is often used in this field to maximize the use of medical data, which is costly and difficult to collect. (e.g.: https://www.nature.com/articles/s41598-021-98374-7)

4. **Natural Language Processing (NLP)**: Text classification models (sentiment analysis, spam detection, article categorization) use simple or stratified splitting depending on whether the classes are balanced or not. Reproducibility via the seed is particularly important in this domain, where reference datasets (*benchmarks*) must be evaluated under strictly identical conditions to allow comparisons between publications.

This list is not exhaustive. Whatever the domain — computer vision, recommendation systems, bioinformatics, or robotics — choosing a splitting method adapted to the nature of the data is an indispensable preliminary step.

## 7. Limitations and Challenges

Although `DataSplitter` covers the most common use cases, it has certain limitations and challenges that should be kept in mind when using it on real problems.

1. **pandas compatibility only**: Unlike scikit-learn, which accepts both NumPy arrays and pandas DataFrames, `DataSplitter` relies exclusively on `.iloc[]` and can therefore only process `pd.DataFrame` and `pd.Series` objects. Using it with raw NumPy arrays would require adapting the code.

2. **Truncation in K-Fold**: When the size of the dataset is not an exact multiple of `k`, the excess observations are simply ignored. For `n=201, k=5`, one observation will systematically be excluded from all evaluation. Scikit-learn resolves this issue in a more elaborate way by distributing the remaining observations across the last folds.

3. **No stratified K-Fold**: The current K-Fold implementation does not include stratification. On heavily imbalanced datasets, some folds could contain no examples from the minority class, making the evaluation locally inoperative. Scikit-learn offers `StratifiedKFold` for this case.

4. **Global seed (potential bias)**: The seed is fixed at the global level via `np.random.seed(seed)` in the constructor, which affects the state of NumPy's pseudo-random number generator for the entire process. In a multi-threaded environment or when using several simultaneous instances of `DataSplitter`, this can produce unexpected interactions. A more robust implementation would use a local generator (`np.random.default_rng(seed)`).

5. **No temporal cross-validation**: For time series requiring robust evaluation (and not just a single split), `DataSplitter` does not offer an equivalent to scikit-learn's `TimeSeriesSplit`. The user will need to turn to scikit-learn for this specific need.

Ultimately, `DataSplitter` is a pedagogical and functional tool, designed to be transparent and understandable. For large-scale production projects, scikit-learn offers additional guarantees in terms of robustness, optimization, and edge case coverage.

## 8. References

- **scikit-learn — model_selection (official documentation)**
  https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection

- **scikit-learn — TimeSeriesSplit**
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html

- **Hastie, T., Tibshirani, R., Friedman, J.** — *The Elements of Statistical Learning* (2nd ed., 2009)
  https://hastie.su.domains/ElemStatLearn/

- **Géron, A.** — *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed., O'Reilly, 2022)

- **Wikipedia — Cross-validation (statistics)**
  https://en.wikipedia.org/wiki/Cross-validation_(statistics)

- **Kaggle — Data Leakage (interactive course)**
  https://www.kaggle.com/code/alexisbcook/data-leakage

- **NumPy — numpy.random documentation**
  https://numpy.org/doc/stable/reference/random/index.html

- **Towards Data Science — Train/Test Split and Cross Validation**
  https://towardsdatascience.com/train-test-split-and-cross-validation-in-python-80b61beca4b6